# Consultor Inteligente de Procedimentos Internos — Notebook de desenvolvimento

Este notebook reune a EDA do dataset sintetico de POPs, a inspecao do modelo de busca (TF-IDF + cosine) e a validacao de casos canonicos. Ele reaproveita os modulos `src/search.py` e `src/llm.py` em vez de reimplementa-los, para que a busca usada aqui seja exatamente a mesma do app e do CLI.

**Sumario**
1. Setup — imports e caminho do dataset
2. EDA — distribuicao das categorias, redundancia, qualidade do texto
3. Treino do modelo — vetorizacao TF-IDF e top termos por IDF
4. Validacao — consultas canonicas (wifi, impressora, software pirata, visita in loco)

In [ ]:
import os
import sys
import pandas as pd

# Permite importar os modulos de src/ a partir deste notebook.
RAIZ = os.path.dirname(os.path.dirname(os.path.abspath('__file__'))) \
    if os.path.exists('__file__') else os.getcwd()
if not RAIZ.endswith('notebooks'):
    # Quando o notebook e aberto via Jupyter, o cwd ja e a raiz do projeto
    RAIZ = os.getcwd()
SRC = os.path.join(RAIZ, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

DATA = os.path.join(RAIZ, 'data', 'dataset_suporte_interno_sintetico.csv.xls')
print('RAIZ:', RAIZ)
print('SRC :', SRC)
print('DATA:', DATA, '-> existe?', os.path.exists(DATA))

In [ ]:
# Dataset pode estar em latin-1 (origem Excel) ou utf-8. Tenta os dois.
try:
    df = pd.read_csv(DATA, sep=';', encoding='latin-1')
except UnicodeDecodeError:
    df = pd.read_csv(DATA, sep=';', encoding='utf-8')

print(f'Linhas: {len(df)}  |  Colunas: {list(df.columns)}')
print()
print('--- Distribuicao por categoria ---')
print(df['Categoria_Problema'].value_counts())
print()
print('--- Redundancia: quantas linhas tem a mesma Regra_POP por categoria? ---')
for cat, grupo in df.groupby('Categoria_Problema'):
    n_linhas = len(grupo)
    n_regras = grupo['Regra_POP'].nunique()
    if n_linhas != n_regras:
        print(f'  {cat}: {n_linhas} linhas, {n_regras} regras unicas')
print()
print('--- Valores nulos por coluna ---')
print(df.isna().sum())
print()
print('--- Tamanho medio (caracteres) de Descricao_Chamado ---')
print(df['Descricao_Chamado'].astype(str).str.len().describe())

In [ ]:
import search

busca = search.SemanticSearch(csv_path=DATA)
print(f'Base disponivel? {busca.disponivel()}')
print(f'Documentos indexados: {busca.total_documentos()}')
print(f'Shape da matriz TF-IDF: {busca.matrix.shape}')
print(f'Bigramas habilitados: ngram_range={busca.vectorizer.ngram_range}, max_features={busca.vectorizer.max_features}')
print()
print('--- Top 15 termos por IDF (mais raros / mais discriminativos) ---')
feature_names = busca.vectorizer.get_feature_names_out()
idfs = busca.vectorizer.idf_
top = sorted(zip(feature_names, idfs), key=lambda x: x[1], reverse=True)[:15]
for termo, idf in top:
    print(f'  {termo:30s}  idf={idf:.3f}')

In [ ]:
casos = [
    'minha impressora travou',
    'nao consigo entrar no wifi da universidade',
    'quero instalar photoshop mas e pago',
    'preciso que um tecnico va ate minha sala',
    'oi, tudo bem?',  # expectativa: metodo 'semantica' mas baixa similaridade
    'xyz123 sem sentido nenhum',  # expectativa: vazio / fallback
]

for c in casos:
    resultados, metodo = busca.consulta_completa(c, top_k=3)
    print(f'\n>>> Consulta: {c!r}  |  metodo={metodo}  |  {len(resultados)} resultados')
    for r in resultados:
        sim = r.get('similarity')
        sim_txt = f'{sim:.3f}' if sim is not None else 'fallback'
        cat = r['row']['Categoria_Problema']
        regra = r['row']['Regra_POP'][:80].replace('\n', ' ')
        print(f'   [{sim_txt}] {cat}: {regra}...')